# 19. 3D point-cloud and detection — connected small pipelines

Only point counts/grid sizes are reduced. Local geometry, hierarchy, BEV construction and 3D detection heads remain connected.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. PointNet and reusable FPS/grouping


In [ ]:
points = torch.tensor([
    [0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0], [1.0, 1.0, 0.2], [0.8, 0.2, 0.7],
    [1.2, 0.7, 0.4], [0.3, 1.1, 0.6],
], device=device)

point_mlp = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 16)).to(device)
point_features = point_mlp(points)
print("PointNet global:", point_features.max(dim=0).values.shape)

def farthest_point_sampling(xyz, count):
    selected = [0]
    minimum = torch.cdist(xyz, xyz[[0]]).squeeze(1)
    for _ in range(count - 1):
        next_id = int(minimum.argmax())
        selected.append(next_id)
        new_distance = torch.cdist(xyz, xyz[[next_id]]).squeeze(1)
        minimum = torch.minimum(minimum, new_distance)
    return torch.tensor(selected, device=xyz.device)

def group_neighbors(query_xyz, source_xyz, k):
    distances = torch.cdist(query_xyz, source_xyz)
    ids = distances.topk(k, largest=False).indices
    return ids, source_xyz[ids] - query_xyz[:, None]


## 2. Two-stage PointNet++ set abstraction


In [ ]:
class SetAbstraction(nn.Module):
    def __init__(self, input_feature_dim, output_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(3 + input_feature_dim, output_dim),
            nn.ReLU(),
            nn.Linear(output_dim, output_dim),
        )

    def forward(self, source_xyz, source_features, centroid_count, neighbors):
        centroid_ids = farthest_point_sampling(source_xyz, centroid_count)
        centroid_xyz = source_xyz[centroid_ids]
        neighbor_ids, relative_xyz = group_neighbors(centroid_xyz, source_xyz, neighbors)
        neighbor_features = source_features[neighbor_ids]
        local_input = torch.cat([relative_xyz, neighbor_features], dim=-1)
        local = self.mlp(local_input).max(dim=1).values
        return centroid_xyz, local


sa1 = SetAbstraction(16, 24).to(device)
sa2 = SetAbstraction(24, 32).to(device)
xyz1, feat1 = sa1(points, point_features, centroid_count=4, neighbors=4)
xyz2, feat2 = sa2(xyz1, feat1, centroid_count=2, neighbors=3)
print("SA1:", xyz1.shape, feat1.shape)
print("SA2:", xyz2.shape, feat2.shape)


## 3. DGCNN EdgeConv and Point Transformer local vector attention


In [ ]:
pairwise = torch.cdist(points, points)
knn_ids = pairwise.topk(k=3, largest=False).indices[:, 1:]
center = points[:, None].expand(-1, knn_ids.size(1), -1)
neighbor = points[knn_ids]
edge_input = torch.cat([center, neighbor - center], dim=-1)
edge_mlp = nn.Sequential(nn.Linear(6, 16), nn.ReLU(), nn.Linear(16, 16)).to(device)
edge_features = edge_mlp(edge_input).max(dim=1).values

q_proj = nn.Linear(16, 16).to(device)
k_proj = nn.Linear(16, 16).to(device)
v_proj = nn.Linear(16, 16).to(device)
pos_mlp = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 16)).to(device)
attn_mlp = nn.Sequential(nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 16)).to(device)

q = q_proj(point_features)
k = k_proj(point_features)[knn_ids]
v = v_proj(point_features)[knn_ids]
relative = points[:, None] - points[knn_ids]
pos = pos_mlp(relative)
logits = attn_mlp(q[:, None] - k + pos)
weights = logits.softmax(dim=1)
point_transformer = (weights * (v + pos)).sum(dim=1)
print("EdgeConv:", edge_features.shape)
print("Point Transformer:", point_transformer.shape)


## 4. Pillar/BEV aggregation


In [ ]:
voxel_size = 0.5
xy = torch.floor(points[:, :2] / voxel_size).long()
xy = xy - xy.min(dim=0).values
height = int(xy[:, 1].max()) + 1
width = int(xy[:, 0].max()) + 1
bev = torch.zeros(16, height, width, device=device)
counts = torch.zeros(1, height, width, device=device)
for index in range(points.size(0)):
    x_id = int(xy[index, 0])
    y_id = int(xy[index, 1])
    bev[:, y_id, x_id] += point_features[index]
    counts[:, y_id, x_id] += 1
bev = bev / counts.clamp_min(1)
print("BEV:", bev.shape)


## 5. CenterPoint-style BEV backbone, prediction heads and oriented 3D box decode


In [ ]:
class TinyCenterPoint(nn.Module):
    def __init__(self, input_channels=16, hidden=24):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(input_channels, hidden, 3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
        )
        self.heatmap = nn.Conv2d(hidden, 1, 1)
        self.offset = nn.Conv2d(hidden, 2, 1)
        self.height = nn.Conv2d(hidden, 1, 1)
        self.dimensions = nn.Conv2d(hidden, 3, 1)
        self.rotation = nn.Conv2d(hidden, 2, 1)

    def forward(self, x):
        h = self.backbone(x)
        return {
            "heatmap": self.heatmap(h).sigmoid(),
            "offset": self.offset(h),
            "height": self.height(h),
            "dimensions": F.softplus(self.dimensions(h)),
            "rotation": self.rotation(h),
        }

def decode_box(prediction, voxel_size=0.5):
    heatmap = prediction["heatmap"]
    batch, _, height, width = heatmap.shape
    score, flat_index = heatmap.view(batch, -1).max(dim=-1)
    y = flat_index // width
    x = flat_index % width
    batch_ids = torch.arange(batch, device=heatmap.device)

    offset = prediction["offset"][batch_ids, :, y, x]
    z = prediction["height"][batch_ids, 0, y, x]
    dims = prediction["dimensions"][batch_ids, :, y, x]
    rot = prediction["rotation"][batch_ids, :, y, x]
    yaw = torch.atan2(rot[:, 0], rot[:, 1])

    center_x = (x.float() + offset[:, 0]) * voxel_size
    center_y = (y.float() + offset[:, 1]) * voxel_size
    box = torch.cat([
        center_x[:, None], center_y[:, None], z[:, None], dims, yaw[:, None]
    ], dim=-1)
    return score, box


centerpoint = TinyCenterPoint().to(device)
prediction = centerpoint(bev[None])
score, box = decode_box(prediction)
loss = sum(value.square().mean() for value in prediction.values())
loss.backward()
print("decoded [x,y,z,w,l,h,yaw]:", box)
print("backbone grad:", centerpoint.backbone[0].weight.grad.norm().item())


## References and provenance

- PointNet++: hierarchical FPS -> grouping -> local PointNet set abstraction.
- DGCNN: local EdgeConv.
- Point Transformer: local neighborhoods plus relative 3D position in attention weights and values.
- CenterPoint: BEV backbone plus center, offset, z, dimensions and rotation heads feeding oriented-box decoding.
